<img src="https://github.com/ArchiColab/sitex/blob/main/images/banner.png?raw=1" width="1000"/>

# Simple Isochrone

**The question:** what would change if there were a new kindergarten here, or a new footpath
there? The objective of this notebook is to investigate how far people can walk from one kind of
facility, on foot and by bike (the two modes of the **15-minute city**, Moreno et al., 2021), and
to test design proposals (a new facility or a new street link) against the situation today.

> Moreno, C., Allam, Z., Chabaud, D., Gall, C., & Pratlong, F. (2021). Introducing the
> "15-Minute City": Sustainability, resilience and place identity in future post-pandemic
> cities. *Smart Cities*, 4(1), 93–111. https://doi.org/10.3390/smartcities4010006

Load a set of facilities, load a street network, and draw the areas you can reach on foot
from the nearest facility in 5, 10 and 15 minutes, and by bike in 5 and 10 minutes. The result is one GeoPackage for a QGIS
layout.

The notebook is optimized for Google Colab. Because it runs in the cloud, it can be used
on machines with limited computing power and in areas with limited bandwidth.

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_overview.svg?raw=1" width="1000"/>

**Library:** [`sitex`](https://github.com/ArchiColab/sitex)

**Requires:** two layers, both prepared and edited in a GIS such as QGIS and saved as GeoPackage.
Nothing is downloaded.
- **A point layer with the facilities** (`POI_FILE`), used exactly as it is: every point counts
  and nothing is filtered. To use only some facilities, select them in QGIS and export them
  first. By default `gdf/healthcare.gpkg`, the healthcare places of
  `03-NA03-Amenity_Isochrones.ipynb`.
- **A line layer with the street network** (`STREETS_FILE`), by default the OSM walk network
  `osm/pleiku_walk.gpkg`. To test a new street, draw it in QGIS in a copy of this layer.

**Both layers must be in the same, metric CRS** (see the next section).

**What you edit:** only the *Your choices* cell in Section 1: which two files to use.

---
## Coordinate system (CRS)

**The street layer and the facility layer must be in the same CRS, and it must be a metric one** (a projected CRS such as a UTM zone, not longitude/latitude). Distances, walking times and areas are computed in the unit of that CRS.

- The notebook **reads the CRS from the street file** and checks the other files against it. You do not type an EPSG code. It does not reproject: if a file has no CRS, is in degrees, or differs from the streets, it stops and names the file.
- **To fix a file in QGIS:** right-click the layer → *Export → Save Features As…* → choose the CRS of the street layer → save as GeoPackage.
- The results are saved in the same CRS, so lengths and areas are in metres in QGIS.
- The workshop files are in the site's UTM zone (global UTM, not VN-2000). To combine them with official VN-2000 data, first reproject that data to this CRS.

---
## 1. Setup & your choices

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_Setup.svg?raw=1" width="1000"/>

Run the first cell once, then set your choices in the second:

| Setting | What it does |
|---|---|
| `STREETS_FILE` | The street network: a line layer, a path inside the data folder. |
| `POI_FILE` | The facilities: a point layer, a path inside the data folder. Every point is used. |
| `TIME_BANDS` | The walking times in minutes. |
| `ISOCHRONE_SHAPE` | `"concave"` (recommended) draws the outline around every street junction you can reach in time, as tight as possible while it stays one shape and contains every street you walked (Dolmajian, 2023). `"convex"` is the smoother convex hull, which also covers corners no street reaches. |
| `BIKE_SPEED_KMH`, `BIKE_TIME_BANDS` | The cycling speed (15 km/h) and the bike times in minutes, for Section 5. The bike uses the same street network as walking. |

The CRS is not a setting: it is read from the street file, and the point file must match it.

In [1]:
#@title <font color=#1B7192> Configuration </font>  { display-mode: "form" }

# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    from google.colab import drive
    drive.mount("/content/gdrive")

    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")

    if not (DATA_DIR / "overture").is_dir():
        raise FileNotFoundError(
            f"{DATA_DIR} not found (or incomplete) in your Google Drive. "
            "Before the workshop: download the workshop data zip and unzip it into "
            "My Drive/Colab_Outputs -- see the workshop setup instructions."
        )
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import geopandas as gpd
from IPython.display import display

from sitex.core.config import slugify
from sitex.network import isochrones as iso

Running in: Local environment


In [2]:
#@title <font color=#1B7192> Your choices </font>  { display-mode: "form" }

# Your two input layers, paths inside the data folder. Edit them in QGIS, save as GeoPackage.
# Both must be in the same metric CRS (checked in Section 2).
STREETS_FILE = "osm/pleiku_walk.gpkg"     # lines: the street network
POI_FILE = "gdf/healthcare.gpkg"          # points: the facilities, every point is used

TIME_BANDS = [5, 10, 15]      # minutes
ISOCHRONE_SHAPE = "concave"   # "concave" (recommended) or "convex"
WALK_SPEED_KMH = 4.8
BIKE_SPEED_KMH = 15.0        # Schuhmacher et al. (2025), see Section 5
BIKE_TIME_BANDS = [5, 10]    # minutes by bike

# Facilities farther than this from any street are set aside, not snapped to a distant one
MAX_SNAP_M = 250

POI_LABEL = Path(POI_FILE).stem.replace("_", " ").capitalize()   # from the file name, used in the output name

---
## 2. Load POIs

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_pois.svg?raw=1" width="1000"/>

First the CRS of both files is read and compared (see *Coordinate system* above); the notebook
stops here if they differ. The points are then used exactly as they are in the file: nothing is
classified, cleaned or filtered. To use only some of them, select and export them in QGIS first.
Points without a `name` get a numbered one.

In [3]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 2. Load POIs ###", flush=True)

print("CRS of the input files:")
LOCAL_EPSG = iso.crs_from_files(DATA_DIR / STREETS_FILE, {"facilities": DATA_DIR / POI_FILE})
print(f"Analysis CRS: EPSG:{LOCAL_EPSG}, read from the street file\n")

facilities = iso.load_points(DATA_DIR / POI_FILE, LOCAL_EPSG, label=POI_LABEL)
print(f"{len(facilities)} points loaded from {POI_FILE}")
if facilities.empty:
    print("No points in the file -- check POI_FILE.")

print(f"--- 2. done in {time.time() - _t0:.2f}s ---\n")
facilities[["name", "geometry"]].head(10)

### 2. Load POIs ###
CRS of the input files:
  streets    pleiku_walk.gpkg: EPSG:32649
  facilities healthcare.gpkg: EPSG:32649
Analysis CRS: EPSG:32649, read from the street file

20 points loaded from gdf/healthcare.gpkg
--- 2. done in 0.33s ---



,name,geometry
0,Bệnh viện Lão Khoa Gia Lai,POINT (177500.706 1546684.679)
1,Tram y tế Phù Đổng,POINT (177591.849 1547084.907)
2,Bệnh Viện Mắt Quốc Tế Sài Gòn - Gia Lai,POINT (176046.853 1547301.018)
3,Trạm Y Tế Phường Hội Thương,POINT (176551.548 1547405.04)
4,Bệnh viện Cao Su Chư Sê,POINT (176184.813 1547222.899)
5,Bệnh Xá Công An Tỉnh Gia Lai,POINT (176315.1 1547293.792)
6,Khoa Phục Hồi Chức Năng Bệnh Viện Đa Khoa Tỉnh...,POINT (178837.7 1547662.707)
7,Bệnh viện Đa khoa Gia Lai,POINT (178867.603 1547813.049)
8,Khoa Giải Phẫu Bệnh - BVĐK tỉnh Gia Lai.,POINT (178898.674 1547817.992)
9,Bệnh Viện Dã Chiến Gia Lai,POINT (178991.652 1547917.265)


---
## 3. Street network

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_network.svg?raw=1" width="1000"/>

The street layer is split wherever two lines cross, so a street you drew in QGIS joins the
existing network where it touches it (turn on snapping while drawing). Each point is then
attached to its nearest street junction; points more than `MAX_SNAP_M` from any street are set
aside.

In [4]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 3. Street network ###", flush=True)

G = iso.network_from_files([DATA_DIR / STREETS_FILE], LOCAL_EPSG, WALK_SPEED_KMH)
print(f"Network: {G.number_of_nodes():,} junctions, {G.number_of_edges() // 2:,} street segments")

facilities_on, facilities_off = iso.snap_facilities(G, facilities, MAX_SNAP_M)
print(f"{len(facilities_on)} points on the network, {len(facilities_off)} set aside (> {MAX_SNAP_M} m from any street)")
if len(facilities_off):
    print(facilities_off[["name", "snap_dist_m"]].to_string(index=False))

print(f"--- 3. done in {time.time() - _t0:.2f}s ---\n")

### 3. Street network ###
Network: 2,687 junctions, 3,364 street segments
20 points on the network, 0 set aside (> 250 m from any street)
--- 3. done in 3.08s ---



---
## 4. Isochrones

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_analysis.svg?raw=1" width="1000"/>

From every facility, the notebook walks outward along the network up to the longest time
band, then merges all facilities: the 10-minute area is everywhere within 10 minutes of the
*nearest* facility.

Each area is drawn as a concave hull around every street junction you can reach in time
(Dolmajian, 2023): as tight as possible while it stays one shape and contains every street
you walked. Where that is not possible it becomes the convex hull. The edge is approximate,
not an exact line. The walk doesn't stop at the last junction reached: where the time runs out halfway along a
street, that point counts too, and so does the facility itself (Dolmajian, 2023).

**Why this shape, and why no buffer.** We tested the shapes on Pleiku's 21,389 building
footprints, for healthcare facilities and for kindergartens. A building counts as truly
reachable when the walk along the streets, plus the short walk from the street to the
building, fits in the time. The concave hull matched best: it includes about 96% of the truly
reachable buildings, and 93–95% of the buildings inside it are truly reachable. A buffer
around the walked streets did worse at every width from 15 m to 100 m (best at 50 m, which
still takes in more buildings that are not reachable), so no buffer is used. The bands are
also merged so that they always nest: the 10-minute area always contains the 5-minute one.

> Dolmajian, A. (2023). *Creating isochrones: what is the optimal way?* Medium.
> https://medium.com/@arthur.dolmajian/creating-isochrones-what-is-the-optimal-way-dfc77a2ca13a

In [5]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 4. Isochrones ###", flush=True)

facility_iso = iso.facility_isochrones(G, facilities_on, TIME_BANDS, shape=ISOCHRONE_SHAPE)
isochrones = iso.dissolve_isochrones(facility_iso)
bands = iso.isochrone_bands(isochrones)
print(bands[["band", "area_ha"]].to_string(index=False))

print(f"--- 4. done in {time.time() - _t0:.2f}s ---\n")
iso.preview_map(bands, facilities_on)

### 4. Isochrones ###
     band  area_ha
  0-5 min    268.2
 5-10 min    540.5
10-15 min    439.1
--- 4. done in 18.15s ---



---
## 5. By bike

The 15-minute city counts both walking and cycling (Moreno et al., 2021). This section rides
the **same street network** by bike, at `BIKE_SPEED_KMH` = 15 km/h: the low end of the median
speeds Schuhmacher et al. (2025) measured for cyclists in Hamburg, 15.0–21.7 km/h depending on
the street. The low end fits Pleiku, where cyclists share the streets with motorbikes and the
plateau has slopes.

**Why 5 and 10 minutes, not 15.** At 15 km/h a 15-minute ride covers about 3.75 km, which
reaches almost the whole ward, so the map would show little. A **5-minute ride (about 1.25 km)
reaches about as far as a 15-minute walk (1.2 km)**: compare the two on the map and in the table.

Alleys, footpaths and steps in the walk network count as rideable, so in places the bike areas
are a little too generous.

> Schuhmacher, L., Wilkes, G., Kagerbauer, M., & Vortisch, P. (2025). Cycling speeds in
> urban traffic. *Findings*. https://doi.org/10.32866/001c.141204

In [6]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 5. By bike ###", flush=True)

G_bike = iso.with_speed(G, BIKE_SPEED_KMH)  # the same streets, ridden at BIKE_SPEED_KMH
bike_facility_iso = iso.facility_isochrones(G_bike, facilities_on, BIKE_TIME_BANDS, shape=ISOCHRONE_SHAPE)
bike_isochrones = iso.dissolve_isochrones(bike_facility_iso)
bike_bands = iso.isochrone_bands(bike_isochrones)
print(bike_bands[["band", "area_ha"]].to_string(index=False))

walk_ha = isochrones.loc[isochrones["time_min"] == max(TIME_BANDS), "area_ha"].iloc[0]
bike_ha = bike_isochrones.loc[bike_isochrones["time_min"] == min(BIKE_TIME_BANDS), "area_ha"].iloc[0]
print(f"\n{max(TIME_BANDS)}-minute walk: {walk_ha} ha   {min(BIKE_TIME_BANDS)}-minute ride: {bike_ha} ha")

print(f"--- 5. done in {time.time() - _t0:.2f}s ---\n")
iso.preview_map(bike_bands, facilities_on, mode="Bike")

### 5. By bike ###
    band  area_ha
 0-5 min   1302.0
5-10 min    734.5

15-minute walk: 1247.9 ha   5-minute ride: 1302.0 ha
--- 5. done in 38.83s ---



---
## 6. Export to QGIS

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_export.svg?raw=1" width="1000"/>

One GeoPackage in `data/gdf/`, in the CRS of your input layers (metres), so lengths and areas are
in metres in QGIS:

| Layer | Contents |
|---|---|
| `isochrone_bands` | Non-overlapping rings, `band` = "0-5 min", "5-10 min", ... **The layer for your layout:** style it as *Categorized* on `band`. |
| `isochrones` | The full areas per time, nested (the 15-minute area contains the 5-minute one). |
| `bike_isochrone_bands`, `bike_isochrones` | The same two layers by bike (`BIKE_TIME_BANDS`). |
| `facilities` | The facilities used, with `snap_dist_m`. |
| `network` | The street network the walks ran on. |

In [7]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 6. Export to QGIS ###", flush=True)

out_path = iso.export_geopackage(
    DATA_DIR / "gdf" / f"simple_isochrones_{slugify(POI_LABEL)}.gpkg",
    {
        "isochrone_bands": bands,
        "isochrones": isochrones,
        "bike_isochrone_bands": bike_bands,
        "bike_isochrones": bike_isochrones,
        "facilities": facilities_on.drop(columns=["node"]),
        "network": iso.network_edges(G),
    },
)
print(f"Saved {out_path}")

print(f"--- 6. done in {time.time() - _t0:.2f}s ---\n")

### 6. Export to QGIS ###
Saved ..\data\gdf\simple_isochrones_healthcare.gpkg
--- 6. done in 0.27s ---



---
## 7. Discussion & field check: testing a proposal

Three short exercises, from the desk to the street:

1. **Isochrone vs. circle.** In QGIS, draw an 800 m circle around one facility (about 10 minutes
   in a straight line) over its 10-minute isochrone. Where are they most different, and why? A
   ravine, a wall, a block without through streets?
2. **Test one proposal.** Add a new facility to a copy of your point file, *or* draw one new street link
   in a copy of the street layer (QGIS, snapping on), set `POI_FILE` or `STREETS_FILE` to the copy and re-run. How many hectares does the 10-minute area gain?
   Which helps more: the new facility or the new link? Compare both modes: does the proposal
   bring people within a 15-minute walk, or only within a 5-minute ride?
3. **"Nearest" hides capacity.** The merged area counts one small kindergarten the same as
   three large ones. What would you need to know to judge whether an area is really served?

| Pattern on the map | What the model predicts | What to check on the ground |
|---|---|---|
| The isochrone is much smaller than the circle on one side | A barrier | Walk the edge: what stops you? A wall, a ravine, a gated compound, a missing bridge? |
| A short new link gains a large area | A small change with a large effect | Is the link possible? Is there already a path people use, a gap in a fence, a worn track? |
| A candidate site for a new facility | New area within reach | Is the plot free? Can it be reached from a quiet street? |

**What to survey at a barrier:** a photo; what kind of barrier it is; whether people already
cross it informally (a worn path, a gap in a fence), and how many in ten minutes.

*Example of a record:* "East edge of the 10-minute kindergarten area, near Street C: 400 m to the
kindergarten in a straight line, 1.2 km on foot, because of a walled compound. A worn path along
its south wall, used by 6 people in 10 minutes. Drawn as a new street in a copy of the
street layer in QGIS, `STREETS_FILE` set to the copy and re-run: the 10-minute area grows from … ha to … ha.

---
## Notes

- **Walking speed is a constant 4.8 km/h**, the speed Transport for London uses for walk times
  (Transport for London, 2015, *Assessing transport connectivity in London*, Table 2.1,
  https://content.tfl.gov.uk/connectivity-assessment-guide.pdf).
  Slopes, road crossings, missing footpaths and heat
  are not taken into account, so real walking times can be longer.
- **Cycling speed is a constant 15 km/h** on the same network (Schuhmacher et al., 2025, see
  Section 5). Slopes and traffic are not taken into account.
- **The edge of each area is approximate.** A hull also covers some corners between streets;
  `"convex"` covers more of them than the default `"concave"`.
- **To compare several kinds of facility**, run the notebook once per point file (for example
  `gdf/healthcare.gpkg`, `gdf/kindergarten.gpkg`): each run writes its own file, named after the
  point file.

---
## When the model is wrong

Where the map and the street disagree, write down why. Common reasons:

- **The bike rides at a constant 15 km/h** on every street of the walk network, including
  alleys, steps and slopes. The map knows nothing about the motorbike traffic that decides
  whether cycling feels safe.
- **Informal paths are missing** from OpenStreetMap, so the real walk is often shorter than the
  map's.
- **Mapped streets can be closed:** a gate locked at night, a private lane, a road without a
  crossing.
- **The edge is a hull,** an approximate outline, not a line you can check to the metre.

The informal shortcuts you find are the best candidates for a new link in your proposal.

## Where to go next

| Next step | Open |
|---|---|
| Which facility serves which part of the ward, and how many buildings | [`03-NA05-Simple_Service_Area.ipynb`](03-NA05-Simple_Service_Area.ipynb) |
| How a new street changes movement across the whole network (scenario test, Section 10) | [`03-NA01-SpaceSyntax.ipynb`](03-NA01-SpaceSyntax.ipynb) |

**Into the design phase:** keep the before/after files of each proposal. The gain in hectares
(or in buildings, with the service-area notebook) is a simple, honest argument for a new
facility or link in your final presentation.